In [ ]:
import os
import re
from typing import List
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

# Load environment variables from .env file (if present)
load_dotenv()


# =====================================================================
# 1. Structured Output Schema (Pydantic)
# =====================================================================
class TranscriptFeatureAudit(BaseModel):
    transcript_rating: int = Field(
        description="Overall discrete rating: strictly 2, 3, or 4 based on transcript evidence."
    )
    matched_rubric_pattern: str = Field(
        description=(
            "The single best-matching rubric statement from internal standards:\n"
            "- For Rating 4:\n"
            "  * 'Questions asked individually by name and Home work was given.'\n"
            "  * 'Good rapport building and asked questions individually.'\n"
            "- For Rating 3:\n"
            "  * 'Included a solid introduction and assigned Homework.'\n"
            "  * 'Good intro and included everyone, but missed Homework.'\n"
            "  * 'Addressed learners during setup and puzzles; gave Homework.'\n"
            "  * 'Maintained one-to-one queries, but should give Homework.'\n"
            "  * 'General interactive flow was present.'\n"
            "- For Rating 2:\n"
            "  * 'Lacked good intro/conclusion and joiner introductions.'\n"
            "  * 'Skipped rapport for revision; missed joiner intros.'\n"
            "  * 'Directly entered class without building rapport.'"
        )
    )
    rapport_building_present: bool = Field(
        description="True if warm greeting, informal check-in, or joiner introduction took place before curriculum."
    )
    new_joiner_intro_given: bool = Field(
        description="True if new students were given explicit time to introduce themselves."
    )
    individual_questions_by_name: bool = Field(
        description="True if tutor called out students individually by their specific names."
    )
    homework_assigned: bool = Field(
        description="True if explicit homework tasks, puzzles, or assignments were given towards the end."
    )
    students_called_by_name: List[str] = Field(
        description="List of distinct student names identified as being addressed in the session."
    )
    summary_observations: str = Field(
        description="A concise 1-2 sentence explanation justifying the assigned rating and pattern match."
    )


# =====================================================================
# 2. System Rubric Prompt
# =====================================================================
RUBRIC_SYSTEM_PROMPT = """
You are an expert pedagogical quality auditor evaluating online children's class session transcripts.
Your task is to analyze the entire class transcript and classify it into Rating 2, 3, or 4 strictly according to the reference patterns below:

### Rating 4 Standards:
- "Questions asked individually by name and Home work was given."
- "Good rapport building and asked questions individually."
Criteria: Tutor establishes warm rapport in opening minutes, consistently nominates learners by individual names, and assigns homework at wrap-up.

### Rating 3 Standards:
- "Included a solid introduction and assigned Homework."
- "Good intro and included everyone, but missed Homework."
- "Addressed learners during setup and puzzles; gave Homework."
- "Maintained one-to-one queries, but should give Homework."
- "General interactive flow was present."
Criteria: Adequate structure and questions; either omits homework, shows moderate warmth, or provides only basic individual engagement.

### Rating 2 Standards:
- "Lacked good intro/conclusion and joiner introductions."
- "Skipped rapport for revision; missed joiner intros."
- "Directly entered class without building rapport."
Criteria: Dives directly into chess moves or revision without icebreakers/rapport, fails to allow new joiners to introduce themselves, or has no proper closure.

Instructions:
1. First 5 to 8 minutes: Check for warm rapport building and whether new joiners were given space to introduce themselves.
2. Middle segment: Verify whether the tutor calls students individually by name rather than addressing the room generically.
3. Final 10 minutes: Verify whether homework (HW) was explicitly given and explained.
4. Select the single best-fitting matched_rubric_pattern from the approved list above.
"""


# =====================================================================
# 3. LangChain Model & Chain Setup
# =====================================================================
def get_auditor_chain():
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError(
            "GEMINI_API_KEY environment variable is missing. "
            "Please set it or add it to a .env file."
        )

    # Use the active flash model endpoint
    llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key,
    temperature=0.0,
)

    structured_llm = llm.with_structured_output(TranscriptFeatureAudit)

    prompt_template = ChatPromptTemplate.from_messages([
        ("system", RUBRIC_SYSTEM_PROMPT),
        ("human", "Class Transcript Content:\n\n{transcript_text}"),
    ])

    return prompt_template | structured_llm


# =====================================================================
# 4. Helper & Audit Execution Functions
# =====================================================================
def clean_vtt_content(raw_vtt: str) -> str:
    """Strips timestamps, cue indices, and WebVTT headers to pass clean dialogue."""
    clean_lines = []
    for line in raw_vtt.splitlines():
        trimmed = line.strip()
        if (
            not trimmed
            or trimmed.startswith("WEBVTT")
            or trimmed.isdigit()
            or "-->" in trimmed
        ):
            continue
        clean_lines.append(trimmed)
    return "\n".join(clean_lines)


def audit_transcript(transcript_text: str) -> TranscriptFeatureAudit:
    """Takes transcript dialogue text, runs analysis, and returns structured audit results."""
    chain = get_auditor_chain()
    return chain.invoke({"transcript_text": transcript_text})


def audit_vtt_file(file_path: str) -> TranscriptFeatureAudit:
    """Reads a .vtt file from disk, cleans it, and executes the audit pipeline."""
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        raw_text = f.read()

    cleaned_text = clean_vtt_content(raw_text)
    return audit_transcript(cleaned_text)




In [ ]:
# =====================================================================
# 5. Execution Example
# =====================================================================
if __name__ == "__main__":
    # Test path to your .vtt file:
    sample_file = "class recordings_\weak\B558 Beginner -transcript-1.1.vtt"

    if os.path.exists(sample_file):
        print(f"Auditing '{sample_file}'...\n" + "=" * 50)
        result = audit_vtt_file(sample_file)

        print(f"Transcript Rating:        {result.transcript_rating}/4")
        print(f"Matched Rubric Pattern:   {result.matched_rubric_pattern}")
        print(f"Rapport Building:         {result.rapport_building_present}")
        print(f"New Joiner Intro:         {result.new_joiner_intro_given}")
        print(f"Individual Queries:       {result.individual_questions_by_name}")
        print(f"Homework Assigned:        {result.homework_assigned}")
        print(f"Students Identified:      {', '.join(result.students_called_by_name)}")
        print(f"Summary:                  {result.summary_observations}")
    else:
        print(f"File '{sample_file}' not found. Please provide a valid file path.")

In [ ]:
import os
from typing import List
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

# 1. Load API Key
load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")

# 2. Define Output Schema
class ClassBinaryAudit(BaseModel):
    rapport_built: bool = Field(
        description="True if tutor established warm intro/rapport before curriculum, False if rushed directly into revision/moves."
    )
    joiner_intro_allowed: bool = Field(
        description="True if new joiners were given explicit space to introduce themselves, False if skipped/missed."
    )
    one_to_one_names: bool = Field(
        description="True if tutor asked questions individually calling students by their specific names."
    )
    homework_given: bool = Field(
        description="True if homework/puzzles/workbook practice was explicitly assigned at the end, False if omitted/forgotten."
    )
    students_called: List[str] = Field(
        description="Names of students addressed individually."
    )

    @property
    def computed_rating(self) -> int:
        """Deterministic score calculation directly from the binary matrix"""
        if self.one_to_one_names and self.homework_given and self.rapport_built:
            return 4
        elif (not self.rapport_built and not self.joiner_intro_allowed) or (
            not self.rapport_built and not self.homework_given
        ):
            return 2
        else:
            return 3

# 3. Rubric Prompt
SYSTEM_PROMPT = """
You are an expert pedagogical auditor. Evaluate the class transcript on 4 strict binary (Yes/No) features:

1. rapport_built (True/False):
   - True: Warm greeting, casual check-ins, or friendly banter before jumping into chess.
   - False: Jumping straight into puzzles/revision without informal rapport.

2. joiner_intro_allowed (True/False):
   - True: Tutor gave new students time to introduce themselves.
   - False: New student was present/acknowledged but no self-introduction was permitted.

3. one_to_one_names (True/False):
   - True: Tutor calls specific students by name for answers (inclusive engagement).
   - False: Open questions to everyone without individual name nominations.

4. homework_given (True/False):
   - True: Tutor explicitly assigned homework/puzzle link before ending.
   - False: Class terminated without homework.

RULES:
- Scan the entire transcript from start to finish.
- Output True or False for each of the 4 features.
"""

# 4. Initialize LLM & Chain
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key,
    temperature=0.0,
)

structured_llm = llm.with_structured_output(ClassBinaryAudit)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Class Transcript:\n\n{transcript_text}"),
])

audit_chain = prompt_template | structured_llm

# 5. The Audit Function
def audit_transcript(transcript_text: str) -> ClassBinaryAudit:
    return audit_chain.invoke({"transcript_text": transcript_text})

In [ ]:
# Read the file
file_path = "class recordings_\weak\B558 Beginner -transcript-1.1.vtt"

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    raw_transcript = f.read()

# Call the function
result: ClassBinaryAudit = audit_transcript(raw_transcript)

# Access individual attributes
print(f"Rapport Built:         {result.rapport_built}")
print(f"Joiner Intro Allowed:  {result.joiner_intro_allowed}")
print(f"One-to-One by Name:    {result.one_to_one_names}")
print(f"Homework Given:        {result.homework_given}")
print(f"Students Called:       {', '.join(result.students_called)}")
print(f"Calculated Rating:     {result.computed_rating}/4")

In [ ]:
import os
from typing import List, Optional
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

load_dotenv()


# =====================================================================
# 1. Binary Audit Schema
# =====================================================================
class ClassBinaryAudit(BaseModel):
    matched_rubric_pattern: str = Field(
        description=(
            "Must be exactly one of the recognized feedback statements:\n"
            "- 'Questions asked individually by name and HW was given.'\n"
            "- 'Good rapport building and asked questions individually.'\n"
            "- 'Included a solid introduction and assigned HW.'\n"
            "- 'Good intro and included everyone, but missed HW.'\n"
            "- 'Addressed learners during setup and puzzles; gave HW.'\n"
            "- 'Maintained one-to-one queries, but should give HW.'\n"
            "- 'Lacked good intro/conclusion and joiner introductions.'\n"
            "- 'Skipped rapport for revision; missed joiner intros.'\n"
            "- 'General interactive flow was present.'\n"
            "- 'Directly entered class without building rapport.'"
        )
    )
    rapport_built: Optional[bool] = Field(
        default=None,
        description=(
            "Rapport / Good Intro?\n"
            "- True (Yes): 'Good rapport building...', 'Included a solid introduction...', 'Good intro...', 'General interactive flow...'\n"
            "- False (No): 'Lacked good intro/conclusion...', 'Skipped rapport for revision...', 'Directly entered class...'\n"
            "- None: If not explicitly evaluated by the feedback statement."
        )
    )
    joiner_intro_allowed: Optional[bool] = Field(
        default=None,
        description=(
            "Joiner Intros Allowed?\n"
            "- False (No): 'Lacked good intro/conclusion and joiner introductions.', 'Skipped rapport for revision; missed joiner intros.'\n"
            "- True (Yes): Dedicated time allowed for new joiner self-introductions.\n"
            "- None: If not explicitly evaluated by the feedback statement."
        )
    )
    one_to_one_names: Optional[bool] = Field(
        default=None,
        description=(
            "One-to-One by Name?\n"
            "- True (Yes): 'Questions asked individually by name...', 'Good rapport building and asked questions individually.', 'Good intro and included everyone...', 'Addressed learners during setup and puzzles...', 'Maintained one-to-one queries...', 'General interactive flow...'\n"
            "- False (No): Open queries only without addressing students individually.\n"
            "- None: If not explicitly evaluated by the feedback statement."
        )
    )
    homework_given: Optional[bool] = Field(
        default=None,
        description=(
            "Homework Given?\n"
            "- True (Yes): '...HW was given.', '...assigned HW.', '...gave HW.'\n"
            "- False (No): '...missed HW.', '...should give HW.'\n"
            "- None: If not explicitly evaluated by the feedback statement."
        )
    )
    students_called: List[str] = Field(
        default_factory=list,
        description="List of distinct student names identified as being addressed individually."
    )
    evidence_notes: str = Field(
        description="Specific dialogue quotes and rationale justifying the extracted features and matched pattern."
    )

    @property
    def computed_rating(self) -> int:
        """Determines rating based on matched ground-truth statements."""
        rating_4_statements = [
            "Questions asked individually by name and HW was given.",
            "Good rapport building and asked questions individually."
        ]
        rating_2_statements = [
            "Lacked good intro/conclusion and joiner introductions.",
            "Skipped rapport for revision; missed joiner intros.",
            "Directly entered class without building rapport."
        ]
        if self.matched_rubric_pattern in rating_4_statements:
            return 4
        elif self.matched_rubric_pattern in rating_2_statements:
            return 2
        return 3


# =====================================================================
# 2. Strict Prompt Grounded in the Binary Table
# =====================================================================
SYSTEM_PROMPT = """
You are an expert pedagogical quality auditor for 8x8 online children's class recordings.
Analyze the provided transcript and evaluate it against this reference mapping table:

| Feedback Statement from Image | Rapport / Good Intro? | Joiner Intros Allowed? | One-to-One by Name? | Homework Given? |
| :--- | :---: | :---: | :---: | :---: |
| "Questions asked individually by name and HW was given." | — | — | Yes | Yes |
| "Good rapport building and asked questions individually." | Yes | — | Yes | — |
| "Included a solid introduction and assigned HW." | Yes | — | — | Yes |
| "Good intro and included everyone, but missed HW." | Yes | — | Yes | No |
| "Addressed learners during setup and puzzles; gave HW." | — | — | Yes | Yes |
| "Maintained one-to-one queries, but should give HW." | — | — | Yes | No |
| "Lacked good intro/conclusion and joiner introductions." | No | No | — | — |
| "Skipped rapport for revision; missed joiner intros." | No | No | — | — |
| "General interactive flow was present." | Yes | — | Yes | — |
| "Directly entered class without building rapport." | No | — | — | — |

AUDIT INSTRUCTIONS:
1. Examine opening minutes and late joiner interactions for rapport building and self-introductions.
2. Review tutor questions during chess puzzles and activities to verify if individual names are called.
3. Check the end of class to determine if homework was assigned.
4. Select the single best-matching 'matched_rubric_pattern' from the table above.
5. Set boolean flags (True for 'Yes', False for 'No', None for '—') aligned with the selected statement.
"""

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.environ.get("GEMINI_API_KEY"),
    temperature=0.0,
)

structured_llm = llm.with_structured_output(ClassBinaryAudit)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Class Transcript:\n\n{transcript_text}"),
])

audit_chain = prompt_template | structured_llm


# =====================================================================
# 3. Execution Helpers
# =====================================================================
def clean_vtt(raw_vtt: str) -> str:
    """Strips timestamps, cue indices, and WebVTT headers to clean the dialogue."""
    lines = []
    for line in raw_vtt.splitlines():
        text = line.strip()
        if not text or text.startswith("WEBVTT") or text.isdigit() or "-->" in text:
            continue
        lines.append(text)
    return "\n".join(lines)


def audit_transcript(transcript_text: str) -> ClassBinaryAudit:
    """Runs binary audit on raw or cleaned transcript text."""
    return audit_chain.invoke({"transcript_text": transcript_text})


def audit_vtt_file(file_path: str) -> ClassBinaryAudit:
    """Reads, cleans, and runs binary audit on a local .vtt file."""
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()
    return audit_transcript(clean_vtt(content))


def format_bool(val: Optional[bool]) -> str:
    if val is True:
        return "Yes"
    elif val is False:
        return "No"
    return "—"




In [ ]:
# =====================================================================
# 4. Usage Example
# =====================================================================
if __name__ == "__main__":
    test_file = "class recordings_\weak\B558 Beginner -transcript-1.1.vtt"

    if os.path.exists(test_file):
        audit: ClassBinaryAudit = audit_vtt_file(test_file)

        print("\n" + "=" * 80)
        print("PEDAGOGICAL TRANSCRIPT AUDIT RESULT")
        print("=" * 80)
        print(f"Matched Statement        : {audit.matched_rubric_pattern}")
        print(f"Computed Rating          : {audit.computed_rating}/4")
        print("-" * 80)
        print(f"Rapport / Good Intro?    : {format_bool(audit.rapport_built)}")
        print(f"Joiner Intros Allowed?   : {format_bool(audit.joiner_intro_allowed)}")
        print(f"One-to-One by Name?      : {format_bool(audit.one_to_one_names)}")
        print(f"Homework Given?          : {format_bool(audit.homework_assigned)}")
        print("-" * 80)
        print(f"Students Addressed       : {', '.join(audit.students_called)}")
        print(f"Evidence & Notes         : {audit.evidence_notes}")
        print("=" * 80)
    else:
        print(f"File '{test_file}' not found. Please provide a valid file path.")

In [ ]:
import os
from typing import List, Literal, Tuple
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# Allowed states per feature
FeatureState = Literal["Yes", "No", "Blank"]

# =====================================================================
# 1. Pydantic LLM Output Schema
# =====================================================================
class FeatureAuditResult(BaseModel):
    # Feature 1: Individual Questioning
    f1_individual_questions: FeatureState = Field(
        description="Yes: asked questions individually / Questions asked individually by name / included everyone / Maintained one to one queries. No: Generic group questions only. Blank: Neutral."
    )
    f1_evidence: str = Field(description="Direct transcript quote or observation.")

    # Feature 2: Homework Assignment
    f2_homework_given: FeatureState = Field(
        description="Yes: HW was given / assigned homework / gave homework. No: missed HW / class ended without homework. Blank: Not evaluated."
    )
    f2_evidence: str = Field(description="Direct transcript quote or observation.")

    # Rapport Building (Embedded in your criteria 2 & Low rules)
    f_rapport_building: FeatureState = Field(
        description="Yes: Good rapport building (informal check-ins, personal banter). No: Skipped rapport / Directly entered class without building rapport. Blank: Minimal/standard greeting."
    )
    f_rapport_evidence: str = Field(description="Direct transcript quote or observation.")

    # Feature 3: Introduction Quality
    f3_solid_intro: FeatureState = Field(
        description="Yes: Included a solid introduction / Good intro. No: Lacked good intro. Blank: Neutral."
    )
    f3_evidence: str = Field(description="Direct transcript quote or observation.")

    # Feature 4: Setup & Puzzles Engagement
    f4_setup_and_puzzles: FeatureState = Field(
        description="Yes: Addressed learners during setup and puzzles. No: Silent or left learners unaddressed during exercises. Blank: No setup/puzzle phase observed."
    )
    f4_evidence: str = Field(description="Direct transcript quote or observation.")

    # Feature 5: General Interactive Flow
    f5_interactive_flow: FeatureState = Field(
        description="Yes: General interactive flow was present (continuous 2-way dialogue). No: Long tutor monologues / passive silence. Blank: Neutral."
    )
    f5_evidence: str = Field(description="Direct transcript quote or observation.")

    # Feature 6: Conclusion Quality
    f6_conclusion_quality: FeatureState = Field(
        description="No: Lacked conclusion (abrupt end, no wrap-up). Yes: Clear wrap-up provided. Blank: Standard end."
    )
    f6_evidence: str = Field(description="Direct transcript quote or observation.")

    # Feature 7: Joiner Introductions
    f7_joiner_intros: FeatureState = Field(
        description="Yes: joiner introductions (allowed time for new student to introduce themselves). No: missed joiner intro / lacked joiner intros. Blank: No new joiners in session."
    )
    f7_evidence: str = Field(description="Direct transcript quote or observation.")

    students_called: List[str] = Field(
        description="Names of individual students called upon during the class."
    )


# =====================================================================
# 2. System Prompt Defining Observable Features
# =====================================================================
EXTRACTION_SYSTEM_PROMPT = """
You are an expert pedagogical auditor for online children's chess classes.
Analyze the provided transcript and extract the state for each of the 7 features strictly as 'Yes', 'No', or 'Blank', providing exact transcript evidence quotes:

1. Feature 1 (Individual Questions):
   - 'Yes': asked questions individually / Questions asked individually by name / included everyone / Maintained one to one queries.
   - 'No': Failed to query individually, addressed only the whole room, or left learners out.
   - 'Blank': Neutral / not evaluated.

2. Feature 2 (Homework):
   - 'Yes': HW was given / assigned homework / gave homework.
   - 'No': missed HW / should give HW (session ended without assigning homework tasks).
   - 'Blank': Not addressed.

3. Rapport Building:
   - 'Yes': Good rapport building (warm casual check-ins, non-chess icebreakers, friendly banter).
   - 'No': Skipped rapport / Directly entered class without building rapport (dived straight into revision/moves).
   - 'Blank': Standard neutral greeting.

4. Feature 3 (Introduction):
   - 'Yes': Included a solid introduction / Good intro (clear orientation of topics/goals).
   - 'No': Lacked good intro.
   - 'Blank': Neutral.

5. Feature 4 (Setup & Puzzles):
   - 'Yes': Addressed learners during setup and puzzles (actively coached learners while setting up or solving).
   - 'No': Passive, left students waiting in silence without coaching.
   - 'Blank': No setup or puzzle phase.

6. Feature 5 (Interactive Flow):
   - 'Yes': General interactive flow was present (active back-and-forth dialogue throughout).
   - 'No': Long monologue by tutor, one-sided interaction.
   - 'Blank': Neutral.

7. Feature 6 (Conclusion):
   - 'No': Lacked conclusion (abrupt drop-off without summarizing or closing).
   - 'Yes': Proper session wrap-up given.
   - 'Blank': Standard exit.

8. Feature 7 (Joiner Introductions):
   - 'Yes': joiner introductions (facilitated dedicated time for new students to introduce themselves).
   - 'No': missed joiner intro / lacked joiner introductions (new student was present but not invited to introduce themselves).
   - 'Blank': No new joiners attended.
"""

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.environ.get("GEMINI_API_KEY"),
    temperature=0.0
)

structured_extractor = llm.with_structured_output(FeatureAuditResult)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", EXTRACTION_SYSTEM_PROMPT),
    ("human", "Class Transcript:\n\n{transcript_text}")
])

extraction_chain = extraction_prompt | structured_extractor


# =====================================================================
# 3. Deterministic Scoring Logic (Your Exact Combinations)
# =====================================================================
def compute_score_and_rule(audit: FeatureAuditResult) -> Tuple[int, str, str]:
    f1 = audit.f1_individual_questions
    f2 = audit.f2_homework_given
    rap = audit.f_rapport_building
    f3 = audit.f3_solid_intro
    f4 = audit.f4_setup_and_puzzles
    f5 = audit.f5_interactive_flow
    f6 = audit.f6_conclusion_quality
    f7 = audit.f7_joiner_intros

    # -----------------------------------------------------------------
    # High Value Feature Combination = 4
    # (Questions asked individually by name + HW was given) OR
    # (Good rapport building + asked questions individually)
    # -----------------------------------------------------------------
    if (f1 == "Yes" and f2 == "Yes") or (rap == "Yes" and f1 == "Yes"):
        if f1 == "Yes" and f2 == "Yes":
            rule = "Questions asked individually by name + HW was given"
        else:
            rule = "Good rapport building + asked questions individually"
        return 4, "High Value", rule

    # -----------------------------------------------------------------
    # Low Value Features Combination = 1
    # (Lacked good intro + lacked conclusion + joiner introductions) OR
    # (Skipped rapport + missed joiner intro) OR
    # (Directly entered class without building rapport.)
    # -----------------------------------------------------------------
    if (
        (f3 == "No" and f6 == "No" and f7 == "Yes")
        or (rap == "No" and f7 == "No")
        or (rap == "No")
    ):
        if f3 == "No" and f6 == "No" and f7 == "Yes":
            rule = "Lacked good intro + lacked conclusion + joiner introductions"
        elif rap == "No" and f7 == "No":
            rule = "Skipped rapport + missed joiner intro"
        else:
            rule = "Directly entered class without building rapport"
        return 1, "Low Value", rule

    # -----------------------------------------------------------------
    # Medium Value Feature Combination = 3
    # (Included a solid introduction + assigned homework) OR
    # (Good intro + included everyone + missed HW) OR
    # (Addressed learners during setup and puzzles + gave homework) OR
    # (Maintained one to one queries + missed Homeworks) OR
    # (General interactive flow was present)
    # -----------------------------------------------------------------
    if f3 == "Yes" and f2 == "Yes":
        rule = "Included a solid introduction + assigned homework"
    elif f3 == "Yes" and f1 == "Yes" and f2 == "No":
        rule = "Good intro + included everyone + missed HW"
    elif f4 == "Yes" and f2 == "Yes":
        rule = "Addressed learners during setup and puzzles + gave homework"
    elif f1 == "Yes" and f2 == "No":
        rule = "Maintained one to one queries + missed Homeworks"
    else:
        rule = "General interactive flow was present"

    return 3, "Medium Value", rule


# =====================================================================
# 4. Helper & Formatter
# =====================================================================
def clean_vtt(raw_text: str) -> str:
    lines = []
    for line in raw_text.splitlines():
        t = line.strip()
        if not t or t.startswith("WEBVTT") or t.isdigit() or "-->" in t:
            continue
        lines.append(t)
    return "\n".join(lines)

def run_pedagogical_audit(vtt_file_path: str):
    with open(vtt_file_path, "r", encoding="utf-8", errors="ignore") as f:
        cleaned = clean_vtt(f.read())

    # Step 1: LLM extracts features
    extracted: FeatureAuditResult = extraction_chain.invoke({"transcript_text": cleaned})

    # Step 2: Deterministic scoring
    score, tier, matched_rule = compute_score_and_rule(extracted)

    # Step 3: Print Table & Score
    print("\n" + "=" * 88)
    print("                      CLASS PEDAGOGICAL FEATURE EXTRACTION TABLE")
    print("=" * 88)
    print(f"{'#':<3} | {'Feature Metric':<32} | {'State':<7} | {'Observation / Quoted Evidence'}")
    print("-" * 88)

    rows = [
        ("1", "Questions Asked Individually", extracted.f1_individual_questions, extracted.f1_evidence),
        ("2", "Homework Assignment", extracted.f2_homework_given, extracted.f2_evidence),
        ("—", "Rapport Building", extracted.f_rapport_building, extracted.f_rapport_evidence),
        ("3", "Introduction Quality", extracted.f3_solid_intro, extracted.f3_evidence),
        ("4", "Setup & Puzzles Engagement", extracted.f4_setup_and_puzzles, extracted.f4_evidence),
        ("5", "General Interactive Flow", extracted.f5_interactive_flow, extracted.f5_evidence),
        ("6", "Session Conclusion", extracted.f6_conclusion_quality, extracted.f6_evidence),
        ("7", "Joiner Introductions", extracted.f7_joiner_intros, extracted.f7_evidence),
    ]

    for num, name, state, ev in rows:
        print(f"{num:<3} | {name:<32} | {state:<7} | {ev}")

    print("=" * 88)
    print(f"Students Addressed Individually : {', '.join(extracted.students_called)}")
    print("-" * 88)
    print(f"OUTPUT SCORE                    : {score}  ({tier})")
    print(f"MATCHED COMBINATION RULE        : \"({matched_rule})\"")
    print("=" * 88 + "\n")

    return extracted, score, matched_rule

In [ ]:
from audit_engine import run_pedagogical_audit

# Pass the path of any transcript file:
extracted_features, score, matched_rule = run_pedagogical_audit("class recordings_\weak\B482 Intermediate -transcript-1.1.vtt")

# Access features directly:
print("Score:", score)
print("Rule:", matched_rule)
print("Questions Individually:", extracted_features.f1_individual_questions)
print("Homework Given:", extracted_features.f2_homework_given)
print("Students Called:", extracted_features.students_called)

# working

In [1]:
import os
from dotenv import load_dotenv

# Load key from .env file or set directly
load_dotenv()

# Verify key exists
if not os.environ.get("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your GEMINI_API_KEY: ")

print("GEMINI_API_KEY configured.")

GEMINI_API_KEY configured.


In [22]:
import os
from dotenv import load_dotenv

# Load key from .env file or set directly
load_dotenv()

# Verify key exists
if not os.environ.get("GROQ_API_KEY"):
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("GROQ_API_KEY configured.")

python-dotenv could not parse statement starting at line 1


GROQ_API_KEY configured.


In [23]:
from typing import List, Literal, Tuple
import pandas as pd
from IPython.display import display, HTML
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

FeatureState = Literal["Yes", "No", "Blank"]

class FeatureAuditResult(BaseModel):
    f1_individual_questions: FeatureState = Field(
        description="Yes: asked questions individually / Questions asked individually by name / included everyone / Maintained one to one queries. No: Generic group questions only. Blank: Neutral."
    )
    f1_evidence: str = Field(description="Direct transcript quote or observation.")

    f2_homework_given: FeatureState = Field(
        description="Yes: HW was given / assigned homework / gave homework. No: missed HW / class ended without homework. Blank: Not evaluated."
    )
    f2_evidence: str = Field(description="Direct transcript quote or observation.")

    f_rapport_building: FeatureState = Field(
        description="Yes: Good rapport building (informal check-ins, personal banter). No: Skipped rapport / Directly entered class without building rapport. Blank: Minimal/standard greeting."
    )
    f_rapport_evidence: str = Field(description="Direct transcript quote or observation.")

    f3_solid_intro: FeatureState = Field(
        description="Yes: Included a solid introduction / Good intro. No: Lacked good intro. Blank: Neutral."
    )
    f3_evidence: str = Field(description="Direct transcript quote or observation.")

    f4_setup_and_puzzles: FeatureState = Field(
        description="Yes: Addressed learners during setup and puzzles. No: Silent or left learners unaddressed during exercises. Blank: No setup/puzzle phase observed."
    )
    f4_evidence: str = Field(description="Direct transcript quote or observation.")

    f5_interactive_flow: FeatureState = Field(
        description="Yes: General interactive flow was present (continuous 2-way dialogue). No: Long tutor monologues / passive silence. Blank: Neutral."
    )
    f5_evidence: str = Field(description="Direct transcript quote or observation.")

    f6_conclusion_quality: FeatureState = Field(
        description="No: Lacked conclusion (abrupt end, no wrap-up). Yes: Clear wrap-up provided. Blank: Standard end."
    )
    f6_evidence: str = Field(description="Direct transcript quote or observation.")

    f7_joiner_intros: FeatureState = Field(
        description="Yes: joiner introductions (allowed time for new student to introduce themselves). No: missed joiner intro / lacked joiner introductions. Blank: No new joiners in session."
    )
    f7_evidence: str = Field(description="Direct transcript quote or observation.")

    students_called: List[str] = Field(
        description="Names of individual students called upon during the class."
    )

EXTRACTION_SYSTEM_PROMPT = """
You are an expert pedagogical auditor for online children's chess classes.
Analyze the provided transcript and extract the state for each of the 7 features strictly as 'Yes', 'No', or 'Blank', providing exact transcript evidence quotes:

1. Feature 1 (Individual Questions):
   - 'Yes': asked questions individually / Questions asked individually by name / included everyone / Maintained one to one queries.
   - 'No': Failed to query individually, addressed only the whole room, or left learners out.
   - 'Blank': Neutral / not evaluated.

2. Feature 2 (Homework):
   - 'Yes': HW was given / assigned homework / gave homework.
   - 'No': missed HW / should give HW (session ended without assigning homework tasks).
   - 'Blank': Not addressed.

3. Rapport Building:
   - 'Yes': Good rapport building (warm casual check-ins, non-chess icebreakers, friendly banter).
   - 'No': Skipped rapport / Directly entered class without building rapport (dived straight into revision/moves).
   - 'Blank': Standard neutral greeting.

4. Feature 3 (Introduction):
   - 'Yes': Included a solid introduction / Good intro (clear orientation of topics/goals).
   - 'No': Lacked good intro.
   - 'Blank': Neutral.

5. Feature 4 (Setup & Puzzles):
   - 'Yes': Addressed learners during setup and puzzles (actively coached learners while setting up or solving).
   - 'No': Passive, left students waiting in silence without coaching.
   - 'Blank': No setup or puzzle phase.

6. Feature 5 (Interactive Flow):
   - 'Yes': General interactive flow was present (active back-and-forth dialogue throughout).
   - 'No': Long monologue by tutor, one-sided interaction.
   - 'Blank': Neutral.

7. Feature 6 (Conclusion):
   - 'No': Lacked conclusion (abrupt drop-off without summarizing or closing).
   - 'Yes': Proper session wrap-up given.
   - 'Blank': Standard exit.

8. Feature 7 (Joiner Introductions):
   - 'Yes': joiner introductions (facilitated dedicated time for new students to introduce themselves).
   - 'No': missed joiner intro / lacked joiner introductions (new student was present but not invited to introduce themselves).
   - 'Blank': No new joiners attended.
"""



In [29]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# 1. Initialize Groq model (Llama-3.3-70b handles full 1-hr transcripts with zero issues)
llm = ChatGroq(
    model_name="qwen/qwen3.8-27b",
    groq_api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0.0
)

# 2. Bind your exact Pydantic schema
structured_extractor = llm.with_structured_output(FeatureAuditResult)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", EXTRACTION_SYSTEM_PROMPT),
    ("human", "Class Transcript:\n\n{transcript_text}")
])

extraction_chain = extraction_prompt | structured_extractor
print("Groq extraction chain initialized successfully.")

Groq extraction chain initialized successfully.


In [33]:
def compute_score_and_rule(audit: FeatureAuditResult) -> Tuple[int, str, str]:
    f1 = audit.f1_individual_questions
    f2 = audit.f2_homework_given
    rap = audit.f_rapport_building
    f3 = audit.f3_solid_intro
    f4 = audit.f4_setup_and_puzzles
    f5 = audit.f5_interactive_flow
    f6 = audit.f6_conclusion_quality
    f7 = audit.f7_joiner_intros

    # High Value Feature Combination = 4
    if (f1 == "Yes" and f2 == "Yes") or (rap == "Yes" and f1 == "Yes"):
        rule = "Questions asked individually by name + HW was given" if (f1 == "Yes" and f2 == "Yes") else "Good rapport building + asked questions individually"
        return 4, "High Value", rule

    # Low Value Features Combination = 1
    if ((f3 == "No" and f6 == "No" and f7 == "Yes") or (rap == "No" and f7 == "No") or (rap == "No")):
        if f3 == "No" and f6 == "No" and f7 == "Yes":
            rule = "Lacked good intro + lacked conclusion + joiner introductions"
        elif rap == "No" and f7 == "No":
            rule = "Skipped rapport + missed joiner intro"
        else:
            rule = "Directly entered class without building rapport"
        return 1, "Low Value", rule

    # Medium Value Feature Combination = 3
    if f3 == "Yes" and f2 == "Yes":
        rule = "Included a solid introduction + assigned homework"
    elif f3 == "Yes" and f1 == "Yes" and f2 == "No":
        rule = "Good intro + included everyone + missed HW"
    elif f4 == "Yes" and f2 == "Yes":
        rule = "Addressed learners during setup and puzzles + gave homework"
    elif f1 == "Yes" and f2 == "No":
        rule = "Maintained one to one queries + missed Homeworks"
    else:
        rule = "General interactive flow was present"

    return 3, "Medium Value", rule

# def clean_vtt(raw_text: str) -> str:
#     lines = []
#     for line in raw_text.splitlines():
#         t = line.strip()
#         if not t or t.startswith("WEBVTT") or t.isdigit() or "-->" in t:
#             continue
#         lines.append(t)
#     return "\n".join(lines)

import re

def clean_and_condense_vtt(raw_text: str, max_dialogue_chars: int = 18000) -> str:
    """
    1. Removes VTT timestamps, line indices, and repetitive filler.
    2. Keeps the critical pedagogical zones:
       - Opening (first ~6,000 chars): rapport, greeting, topic intro, joiners
       - Closing (last ~6,000 chars): homework, wrap-up, conclusion
       - Middle sample (~6,000 chars): questions by name, puzzle setup
    """
    lines = []
    seen = set()
    
    for line in raw_text.splitlines():
        t = line.strip()
        # Drop VTT headers, timing lines, and blank rows
        if not t or t.startswith("WEBVTT") or t.isdigit() or "-->" in t:
            continue
        # Deduplicate consecutive audio stutter / identical caption ticks
        if len(lines) > 0 and lines[-1] == t:
            continue
        lines.append(t)

    full_dialogue = "\n".join(lines)
    
    # If the file is already small enough, pass it directly
    if len(full_dialogue) <= max_dialogue_chars:
        return full_dialogue

    # Window slicing targeted at the 7 features:
    opening = full_dialogue[:7000]       # Rapport, Intro, Joiner Intros
    closing = full_dialogue[-6000:]      # Homework, Conclusion
    
    # Middle window where puzzles & individual turn-taking happen
    mid_start = len(full_dialogue) // 2 - 2500
    mid_end = mid_start + 5000
    middle = full_dialogue[mid_start:mid_end]

    condensed = (
        "=== CLASS OPENING PHASE (MINUTES 0-12) ===\n"
        + opening
        + "\n\n=== MID-CLASS INTERACTIVE & PUZZLE PHASE ===\n"
        + middle
        + "\n\n=== CLASS CLOSING & HOMEWORK PHASE (LAST 10 MINUTES) ===\n"
        + closing
    )
    return condensed

In [ ]:
def audit_and_display(vtt_file_path: str):
    with open(vtt_file_path, "r", encoding="utf-8", errors="ignore") as f:
        condensed_text = clean_and_condense_vtt(f.read())
        
    print(f"Compressed transcript size: {len(condensed_text)} characters (~{len(condensed_text)//4} tokens).")
    
    extracted: FeatureAuditResult = extraction_chain.invoke({"transcript_text": condensed_text})
    score, tier, rule = compute_score_and_rule(extracted)
    
    # Build DataFrame
    table_data = [
        {"#": "1", "Feature Metric": "Questions Asked Individually", "State": extracted.f1_individual_questions, "Observation / Evidence": extracted.f1_evidence},
        {"#": "2", "Feature Metric": "Homework Assignment", "State": extracted.f2_homework_given, "Observation / Evidence": extracted.f2_evidence},
        {"#": "—", "Feature Metric": "Rapport Building", "State": extracted.f_rapport_building, "Observation / Evidence": extracted.f_rapport_evidence},
        {"#": "3", "Feature Metric": "Introduction Quality", "State": extracted.f3_solid_intro, "Observation / Evidence": extracted.f3_evidence},
        {"#": "4", "Feature Metric": "Setup & Puzzles Engagement", "State": extracted.f4_setup_and_puzzles, "Observation / Evidence": extracted.f4_evidence},
        {"#": "5", "Feature Metric": "General Interactive Flow", "State": extracted.f5_interactive_flow, "Observation / Evidence": extracted.f5_evidence},
        {"#": "6", "Feature Metric": "Session Conclusion", "State": extracted.f6_conclusion_quality, "Observation / Evidence": extracted.f6_evidence},
        {"#": "7", "Feature Metric": "Joiner Introductions", "State": extracted.f7_joiner_intros, "Observation / Evidence": extracted.f7_evidence},
    ]
    df = pd.DataFrame(table_data)
    
    # Color badge mapping
    def highlight_state(val):
        if val == "Yes":
            return "background-color: #e2efda; color: #276a3c; font-weight: bold; text-align: center;"
        elif val == "No":
            return "background-color: #fce4d6; color: #c65911; font-weight: bold; text-align: center;"
        return "color: #7f8c8d; text-align: center;"
    
    styled_df = df.style.map(highlight_state, subset=["State"]).set_properties(**{
    'text-align': 'left',
    'white-space': 'normal'
        })
    
    score_color = "#27ae60" if score == 4 else ("#e67e22" if score == 3 else "#c0392b")
    
    header_html = f"""
    <div style="border-left: 5px solid {score_color}; padding: 10px 15px; margin-bottom: 12px; background: #fdfdfd; border-radius: 4px; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
        <h3 style="margin: 0; color: #2c3e50;">File: <span style="font-weight: normal;">{os.path.basename(vtt_file_path)}</span></h3>
        <p style="margin: 5px 0 0 0; font-size: 15px;">
            <b>Final Score:</b> <span style="color: {score_color}; font-size: 18px; font-weight: bold;">{score} / 4</span> ({tier}) &nbsp;|&nbsp; 
            <b>Matched Rule:</b> <i>"{rule}"</i>
        </p>
        <p style="margin: 4px 0 0 0; font-size: 13px; color: #555;">
            <b>Students Addressed:</b> {', '.join(extracted.students_called) if extracted.students_called else 'None Identified'}
        </p>
    </div>
    """
    
    display(HTML(header_html))
    display(styled_df)
    return extracted, score, rule

## Weak

In [32]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\weak\B558 Beginner -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3.8-27b` in organization `org_01kke5ybsyejnt9a6kjzkg7fzj` service tier `on_demand` on input tokens per minute (ITPM): Limit 7000, Requested 12242, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [28]:
import os
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
models = client.models.list()

print("Available Groq Models:")
for model in sorted(models.data, key=lambda x: x.id):
    print(f" - {model.id}")

Available Groq Models:
 - allam-2-7b
 - canopylabs/orpheus-arabic-saudi
 - canopylabs/orpheus-v1-english
 - meta-llama/llama-prompt-guard-2-22m
 - meta-llama/llama-prompt-guard-2-86m
 - openai/gpt-oss-120b
 - openai/gpt-oss-20b
 - openai/gpt-oss-safeguard-20b
 - qwen/qwen3.8-27b
 - whisper-large-v3
 - whisper-large-v3-turbo


In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\weak\B482 Intermediate -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

# quality issue

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\quality issue_\B441 Beginner-transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\quality issue_\B392 Beginner-transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

# Good

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\good\B552 Beginner -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\good\B435 intermediate Part  -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

# excellent

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\excellent\B491 Beginner -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\excellent\B472 intermediate -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

# demo

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\demo_\demo malayalam transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\demo_\demo english transcript.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

# average

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\average_\B516 Beginner-transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = r"class recordings_\average_\B516 Beginner-transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")